# HRAI API demo notebook

In [ ]:
# from main import QueryRequest
!pip install --upgrade pip
!pip install -r requirements.txt

!git lfs pull

In [1]:
import os, json, random, textwrap, requests
from pathlib import Path

from config import conf
from load import get_encoder
from query import query_type, extract_from_resume
from suggestions import get_expanded_skills, get_domain_reports
from pos_extraction import text_to_ngrams

from dotenv import load_dotenv
import faiss
from IPython.display import HTML, display

BASE_URL = 'http://127.0.0.1:8000'

## konfigurace modelu a metadat pro lokální volání
\*pro rychlejší fungování encoderu doporučuji nastavit HF_TOKEN v .env

In [3]:
load_dotenv()
db = faiss.read_index(os.path.join(conf.db_dir, f"all.index"))
with open(os.path.join(conf.data_dir, f"key_to_ent.json"),'r') as f:
    metadata = json.loads(f.read())
model = get_encoder()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [5]:
def _escape(text: str) -> str:
    return (
        text.replace("&", "&amp;")
        .replace("<", "&lt;")
        .replace(">", "&gt;")
        .replace('"', "&quot;")
        .replace("'", "&#39;")
    )


def _badge(label: str, tone: str = "neutral") -> str:
    colors = {
        "neutral": ("#4b5563", "#e5e7eb"),
        "success": ("#047857", "#d1fae5"),
        "info": ("#1d4ed8", "#dbeafe"),
        "warn": ("#b45309", "#fef3c7"),
    }
    fg, bg = colors.get(tone, colors["neutral"])
    return (
        f"<span style='display:inline-block;margin-left:8px;"
        f"padding:2px 8px;border-radius:999px;"
        f"background:{bg};color:{fg};font-size:12px;'>"
        f"{_escape(label)}"
        "</span>"
    )


def _card(html: str) -> str:
    return (
        "<div style='border:1px solid #e5e7eb;border-radius:8px;"
        "padding:12px 16px;margin:8px 0;background:#ffffff;'>"
        f"{html}"
        "</div>"
    )


def _pre_card(text: str, title: str | None = None) -> str:
    heading = f"<div style='font-weight:600;margin-bottom:6px;'>{_escape(title)}</div>" if title else ""
    return _card(f"{heading}<pre style='margin:0;white-space:pre-wrap;'>{_escape(text)}</pre>")


def print_skill(s):
    label = _escape(s.get("label", ""))
    desc = _escape(s.get("description", ""))
    body = f"<div style='font-weight:600;'>{label}</div>"
    if desc:
        body += f"<div style='color:#4b5563;margin-top:4px;'>informace: {desc}</div>"
    display(HTML(_card(body)))


def print_occupation(occ):
    label = _escape(occ.get("label", ""))
    score = occ.get("cosine_score")
    score_text = f"skore: {round(score, 5)}" if score is not None else "skore: n/a"
    desc = _escape(occ.get("description", ""))
    body = (
        f"<div style='font-weight:700;font-size:16px;'>"
        f"{label}"
        f"{_badge(score_text, 'info')}"
        "</div>"
    )
    if desc:
        body += f"<div style='color:#4b5563;margin-top:6px;'>info: {desc}</div>"
    display(HTML(_card(body)))


def print_missing_skills(skills):
    for s in skills:
        if "missing_skills" in s and len(s["missing_skills"]) == 0:
            continue
        if "occupation" in s:
            print_occupation(s["occupation"])
        if "missing_skills" not in s:
            continue
        essential = [item for item in s["missing_skills"] if item["relation_type"] == "essential"]
        optional = [item for item in s["missing_skills"] if item["relation_type"] == "optional"]

        section = "<div style='margin-top:6px;'>"
        if essential:
            section += "<div style='font-weight:600;margin:6px 0 4px;'>Základní dovednosti/znalosti pro výkon pozice:</div>"
            for e in essential:
                section += (
                    "<div style='margin:2px 0;'>"
                    f"{_escape(e['label'])}"
                    "</div>"
                )
        if optional:
            section += "<div style='font-weight:600;margin:10px 0 4px;'>Další užitečné dovednosti:</div>"
            for o in optional:
                section += (
                    "<div style='margin:2px 0;'>"
                    f"{_escape(o['label'])}"
                    "</div>"
                )
        section += "</div>"
        display(HTML(_card(section)))


def print_domain_info(d):
    for occ in d["occupations"]:
        print_occupation(occ["occupation"])
        if "missing_skills" in occ and len(occ["missing_skills"]) > 0:
            print_missing_skills([occ])

## převedení textu na ngramy

In [6]:
resumes = json.loads(Path(os.path.join('testfiles', 'cs_resumes.json')).read_text(encoding='utf-8'))
text = random.choice(resumes)
display(HTML(_pre_card(textwrap.shorten(text, width=2000), "sample resume text")))

In [7]:
ngrams = text_to_ngrams(text)
text_len=len(str.split(text))
display(
    HTML(
        _pre_card(
            f"ngramy: {len(ngrams)}\n"+
            f"slova: {text_len}\n\n"+
            f"{"\n".join(ngrams)}",
            )))

## API endpointy: /resume/skills a /resume/domains (pouze PDF)

In [ ]:
def get_skills(filename: str):
    file_path = Path(os.path.join('testfiles',filename))
    files = {'file': (file_path.name, file_path.read_bytes(), 'application/pdf')}
    res = requests.post(f"{BASE_URL}/resume/skills", files=files)

    display(HTML(_card(
        f"<div style='font-weight:600;'>file: {_escape(filename)}"
        f"{_badge(f'status: {res.status_code}', 'success' if res.ok else 'warn')}"
        "</div>"
    )))
    print_missing_skills(res.json())


def get_domains(filename:str):
    file_path = Path(os.path.join('testfiles',filename))
    files = {'file': (file_path.name, file_path.read_bytes(), 'application/pdf')}
    res = requests.post(f"{BASE_URL}/resume/domains", files=files)

    display(HTML(_card(
        f"<div style='font-weight:600;'>file: {_escape(filename)}"
        f"{_badge(f'status: {res.status_code}', 'success' if res.ok else 'warn')}"
        "</div>"
    )))

    for d in res.json():
        count_badge = _badge(f"shody: {d['occ_count']}", "neutral")
        header = (
            f"<div style='font-weight:700;font-size:16px;'>oblast: {_escape(d['domain'])}" 
            f"{count_badge}"
            "</div>"
        )
        display(HTML(_card(header)))
        if 'occupations' not in d: continue
        if len(d['occupations']) <1: continue
        print_domain_info(d)


In [ ]:
get_skills('resume.pdf')

In [ ]:
get_domains('resume.docx')

## API endpointy: /text/skills a /text/domains

In [ ]:
text_req = {
    'occupations': ['software developer', 'data analyst'],
    'skills': ['python', 'strojové učení', 'sql databáze']
}
skills_resp = requests.post(f"{BASE_URL}/text/skills", json=text_req)
display(HTML(_card(
    f"<div style='font-weight:600;'>endpoint: /text/skills"
    f"{_badge(f'status: {skills_resp.status_code}', 'success' if skills_resp.ok else 'warn')}"
    "</div>"
)))
print_missing_skills(skills_resp.json())


domains_resp = requests.post(f"{BASE_URL}/text/domains", json=text_req)
display(HTML(_card(
    f"<div style='font-weight:600;'>endpoint: /text/domains"
    f"{_badge(f'status: {domains_resp.status_code}', 'success' if domains_resp.ok else 'warn')}"
    "</div>"
)))
for d in domains_resp.json():
    count_badge = _badge(f"shody: {d['occ_count']}", "neutral")
    header = (
        f"<div style='font-weight:700;font-size:16px;'>oblast: {_escape(d['domain'])}" 
        f"{count_badge}"
        "</div>"
    )
    display(HTML(_card(header)))
    if 'occupations' not in d or len(d['occupations']) < 1: continue
    print_domain_info(d)

## API endpoint: /query (každý typ dotazu)

In [ ]:
from main import QueryRequest
# TODO fix example request

query = QueryRequest(
    query='manažerka v mcdonalds',
    query_type='occupation',
    min_set_score=0.7
)

resp = requests.post(f"{BASE_URL}/query", json=query.model_dump())
display(HTML(_card(
    f"<div style='font-weight:600;'>endpoint: /query"
    f"{_badge(f'status: {resp.status_code}', 'success' if resp.ok else 'warn')}"
    "</div>"
)))
display(HTML(_card(
    f"<pre style='margin:0;white-space:pre-wrap;'>"
    f"{_escape(str(resp.json()))}"
    "</pre>"
)))

## Přímé volání query_type pro všechny ent_type

In [ ]:
entry = ['rybář']
ent_type = 'occupation'

results = query_type(db=db, metadata=metadata, model=model, ents=entry, ent_type=ent_type, min_score=0.5)
display(HTML(_card(
    f"<div style='font-weight:600;'>"
    f"{_escape(str(entry))} ({_escape(ent_type)})"
    "</div>"
)))
display(HTML(_card(
    f"<pre style='margin:0;white-space:pre-wrap;'>"
    f"{_escape(str(results))}"
    "</pre>"
)))

## Pipeline: extract_from_resume -> get_expanded_skills -> get_domain_reports

In [ ]:
random_resume = random.choice(resumes)
display(HTML(_pre_card(textwrap.shorten(random_resume, width=2000, placeholder="..."), "sample text")))

extracted_entities = extract_from_resume(db, metadata, model, random_resume)
display(HTML(_card(
    f"<div style='font-weight:600;'>extracted entities</div>"
    f"<pre style='margin:6px 0 0;white-space:pre-wrap;'>"
    f"{_escape(textwrap.shorten(str(extracted_entities), width=2000, placeholder='...'))}"
    "</pre>"
)))

skill_suggestions = get_expanded_skills(metadata, extracted_entities)
display(HTML(_card(
    f"<div style='font-weight:600;'>skill suggestions</div>"
    f"<pre style='margin:6px 0 0;white-space:pre-wrap;'>"
    f"{_escape(textwrap.shorten(str(skill_suggestions), width=2000, placeholder='...'))}"
    "</pre>"
)))

domain_reports = get_domain_reports(skill_suggestions)
display(HTML(_card(
    f"<div style='font-weight:600;'>domain stats</div>"
    f"<pre style='margin:6px 0 0;white-space:pre-wrap;'>"
    f"{_escape(textwrap.shorten(str(domain_reports), width=2000, placeholder='...'))}"
    "</pre>"
)))
